# MCQ Prompt-Format Search: Finding a Format All Models Can Do Above Chance

Earlier tests found that LLaVA-1.5-7B, LLaVA-1.6-Vicuna-7B, and Gemma-3-4B (base+instruct)
all score **at or below chance (25%)** on the 4-way forced-choice MCQ task used for causal
steering evaluation, using a fairly verbose/formal prompt:

> "Which of the following is shown in this image? A) cat  B) dog  C) fish  D) bird.
> Answer with only the letter."

This notebook searches over **much simpler, shorter MCQ prompt formats** to find one where
all four checkpoints can do the underlying visual-discrimination task above chance
(25% for 4-way). If we can't find such a format, MCQ isn't a viable causal-evaluation
protocol for these families at all — but the original prompt might simply be too complex
(too many clauses, an awkward "Which of the following..." construction) for less
instruction-tuned or weaker models to parse correctly, independent of their actual visual
ability.

In [1]:
import sys, re
from pathlib import Path
REPO_ROOT = Path("/mnt/abka03/Projects/vis-head")
sys.path.insert(0, str(REPO_ROOT))

import torch
import numpy as np
import pandas as pd
from transformers import (AutoProcessor, LlavaForConditionalGeneration,
                           LlavaNextForConditionalGeneration, Gemma3ForConditionalGeneration)

from vis_head.imagenet_grid import DEFAULT_IMAGENET_ROOT, list_val_class_dirs, load_class_names, sample_grid

ROWS, COLS = 2, 2
N_CELLS = ROWS * COLS
CELL_SIZE = 256
N_SAMPLES = 24
SEED = 555
OPTION_LETTERS = ["A", "B", "C", "D"]
DEVICE = "cuda:0"

imagenet_class_dirs = list_val_class_dirs(DEFAULT_IMAGENET_ROOT)
imagenet_class_names = load_class_names(DEFAULT_IMAGENET_ROOT)
print(f"{len(imagenet_class_dirs)} ImageNet classes available")

1000 ImageNet classes available


In [2]:
def build_sample(rng):
    grid = sample_grid(rows=ROWS, cols=COLS, cell_size=CELL_SIZE, rng=rng,
                        class_dirs=imagenet_class_dirs, class_names=imagenet_class_names)
    target_cell = int(rng.randint(N_CELLS))
    correct_name = grid.cell_names[target_cell]
    other_names = [n for i, n in enumerate(grid.cell_names) if i != target_cell]
    distractor_idx = rng.choice(len(other_names), size=3, replace=False)
    distractors = [other_names[i] for i in distractor_idx]
    options = [correct_name] + distractors
    order = rng.permutation(len(options))
    options = [options[i] for i in order]
    correct_letter = OPTION_LETTERS[int(np.where(order == 0)[0][0])]
    return grid, options, correct_letter


def parse_letter(text):
    match = re.search(r"\b([ABCD])\b", text.upper())
    return match.group(1) if match else None


# Same fixed set of samples reused across every prompt variant and every model,
# so accuracy differences reflect the prompt/model, not sampling noise.
sample_rng = np.random.RandomState(SEED)
FIXED_SAMPLES = [build_sample(sample_rng) for _ in range(N_SAMPLES)]
print(f"Built {len(FIXED_SAMPLES)} fixed evaluation samples")

Built 24 fixed evaluation samples


## Prompt-format candidates

From verbose/formal (the original) down to as short and plain as possible.

In [3]:
PROMPT_VARIANTS = {
    "original_verbose": lambda opts: (
        "Which of the following is shown in this image? "
        + "  ".join(f"{l}) {n}" for l, n in zip(OPTION_LETTERS, opts))
        + ". Answer with only the letter."
    ),
    "short_question": lambda opts: (
        "What is in the image? "
        + "  ".join(f"{l}) {n}" for l, n in zip(OPTION_LETTERS, opts))
        + " Answer with one letter."
    ),
    "newline_list": lambda opts: (
        "What is in the image?\n"
        + "\n".join(f"{l}. {n}" for l, n in zip(OPTION_LETTERS, opts))
        + "\nAnswer:"
    ),
    "bare_options": lambda opts: (
        " / ".join(f"{l}={n}" for l, n in zip(OPTION_LETTERS, opts))
        + "? Answer:"
    ),
    "minimal": lambda opts: (
        "Image: " + ", ".join(f"{l}) {n}" for l, n in zip(OPTION_LETTERS, opts)) + "?"
    ),
}
for name, fn in PROMPT_VARIANTS.items():
    print(f"--- {name} ---")
    print(fn(["cat", "dog", "fish", "bird"]))
    print()

--- original_verbose ---
Which of the following is shown in this image? A) cat  B) dog  C) fish  D) bird. Answer with only the letter.

--- short_question ---
What is in the image? A) cat  B) dog  C) fish  D) bird Answer with one letter.

--- newline_list ---
What is in the image?
A. cat
B. dog
C. fish
D. bird
Answer:

--- bare_options ---
A=cat / B=dog / C=fish / D=bird? Answer:

--- minimal ---
Image: A) cat, B) dog, C) fish, D) bird?



In [4]:
def run_prompt_search(model, processor, model_kind, use_instruct_template_text_from=None):
    """Evaluate every prompt variant on the same fixed samples for one model."""
    template_processor = use_instruct_template_text_from or processor
    results = {}
    for variant_name, prompt_fn in PROMPT_VARIANTS.items():
        correct = 0
        unparsed = 0
        shown = []
        for grid, options, correct_letter in FIXED_SAMPLES:
            prompt = prompt_fn(options)
            messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}]
            text = template_processor.apply_chat_template([messages], tokenize=False, add_generation_prompt=True)
            text = text[0] if isinstance(text, list) else text
            inputs = processor(text=text, images=[grid.grid], return_tensors="pt").to(DEVICE)
            prompt_length = int(inputs["input_ids"].shape[1])
            with torch.no_grad():
                out = model.generate(**inputs, max_new_tokens=8, do_sample=False)
            gen_ids = out[0][prompt_length:]
            gen_text = processor.tokenizer.decode(gen_ids, skip_special_tokens=True)
            pred = parse_letter(gen_text)
            if pred is None:
                unparsed += 1
            elif pred == correct_letter:
                correct += 1
            if len(shown) < 3:
                shown.append((correct_letter, gen_text))
        acc = correct / len(FIXED_SAMPLES)
        results[variant_name] = {"accuracy": acc, "unparsed": unparsed, "examples": shown}
        print(f"  [{model_kind}] {variant_name:16s} acc={acc:.3f}  unparsed={unparsed}/{len(FIXED_SAMPLES)}  "
              f"examples={[(c, t) for c, t in shown]}")
    return results


all_results = {}

## LLaVA-1.5-7B

In [5]:
proc = AutoProcessor.from_pretrained("llava-hf/llava-1.5-7b-hf")
model = LlavaForConditionalGeneration.from_pretrained(
    "llava-hf/llava-1.5-7b-hf", torch_dtype=torch.bfloat16, attn_implementation="eager"
).to(DEVICE)
model.eval()
all_results["llava-1.5-7b"] = run_prompt_search(model, proc, "llava-1.5-7b")
del model
torch.cuda.empty_cache()

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

  [llava-1.5-7b] original_verbose acc=0.458  unparsed=0/24  examples=[('A', 'A'), ('A', 'D'), ('C', 'A) home theater')]


  [llava-1.5-7b] short_question   acc=0.375  unparsed=0/24  examples=[('A', 'A'), ('A', 'A'), ('C', 'A')]


  [llava-1.5-7b] newline_list     acc=0.292  unparsed=0/24  examples=[('A', 'A. ptarmigan'), ('A', 'A. speedboat'), ('C', 'A')]


  [llava-1.5-7b] bare_options     acc=0.292  unparsed=0/24  examples=[('A', 'A'), ('A', 'A'), ('C', 'A')]


  [llava-1.5-7b] minimal          acc=0.292  unparsed=0/24  examples=[('A', 'A) Ptarmigan\nB'), ('A', 'A) speedboat, B)'), ('C', 'A) home theater, B)')]


## LLaVA-1.6-Vicuna-7B (instruct)

In [6]:
proc = AutoProcessor.from_pretrained("llava-hf/llava-v1.6-vicuna-7b-hf")
model = LlavaNextForConditionalGeneration.from_pretrained(
    "llava-hf/llava-v1.6-vicuna-7b-hf", torch_dtype=torch.bfloat16, attn_implementation="eager"
).to(DEVICE)
model.eval()
all_results["llava-1.6-vicuna-7b"] = run_prompt_search(model, proc, "llava-1.6-vicuna-7b")
del model
torch.cuda.empty_cache()

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/687 [00:00<?, ?it/s]

  [llava-1.6-vicuna-7b] original_verbose acc=0.417  unparsed=0/24  examples=[('A', 'A'), ('A', 'A'), ('C', 'C')]


  [llava-1.6-vicuna-7b] short_question   acc=0.208  unparsed=0/24  examples=[('A', 'B'), ('A', 'B'), ('C', 'C')]


  [llava-1.6-vicuna-7b] newline_list     acc=0.333  unparsed=0/24  examples=[('A', 'A'), ('A', 'A'), ('C', 'A')]


  [llava-1.6-vicuna-7b] bare_options     acc=0.250  unparsed=4/24  examples=[('A', "The image you've provided is a"), ('A', "The image you've provided shows a"), ('C', "The image you've provided appears to")]


  [llava-1.6-vicuna-7b] minimal          acc=0.292  unparsed=0/24  examples=[('A', 'A) ptarmigan\nB)'), ('A', 'A) speedboat.'), ('C', 'A) home theater.')]


## Gemma-3-4B-it (instruct)

Tested first since it has its own working chat template; base model reuses this
template's rendered text below (base has no chat template of its own).

In [7]:
gemma_it_proc = AutoProcessor.from_pretrained("google/gemma-3-4b-it")
model = Gemma3ForConditionalGeneration.from_pretrained(
    "google/gemma-3-4b-it", torch_dtype=torch.bfloat16, attn_implementation="eager"
).to(DEVICE)
model.eval()
all_results["gemma-3-4b-it"] = run_prompt_search(model, gemma_it_proc, "gemma-3-4b-it")
del model
torch.cuda.empty_cache()

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

  [gemma-3-4b-it] original_verbose acc=0.250  unparsed=0/24  examples=[('A', 'C'), ('A', 'D'), ('C', 'C')]


  [gemma-3-4b-it] short_question   acc=0.333  unparsed=0/24  examples=[('A', 'C'), ('A', 'D'), ('C', 'C')]


  [gemma-3-4b-it] newline_list     acc=0.042  unparsed=18/24  examples=[('A', "Let's analyze each image to determine"), ('A', 'The correct answer is **D. lor'), ('C', "Let's analyze the image to determine")]


  [gemma-3-4b-it] bare_options     acc=0.042  unparsed=23/24  examples=[('A', "Here's the breakdown of the images"), ('A', "Here's the breakdown of the images"), ('C', "Here's the breakdown of the images")]


  [gemma-3-4b-it] minimal          acc=0.042  unparsed=23/24  examples=[('A', "Here's the breakdown of the images"), ('A', "Here's the breakdown of the images"), ('C', "Here's the breakdown of what each")]


## Gemma-3-4B-pt (base)

In [8]:
gemma_pt_proc = AutoProcessor.from_pretrained("google/gemma-3-4b-pt")
model = Gemma3ForConditionalGeneration.from_pretrained(
    "google/gemma-3-4b-pt", torch_dtype=torch.bfloat16, attn_implementation="eager"
).to(DEVICE)
model.eval()
all_results["gemma-3-4b-pt"] = run_prompt_search(
    model, gemma_pt_proc, "gemma-3-4b-pt", use_instruct_template_text_from=gemma_it_proc)
del model
torch.cuda.empty_cache()

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

  [gemma-3-4b-pt] original_verbose acc=0.000  unparsed=24/24  examples=[('A', '\n\nWhich of the following is shown'), ('A', '\n\nWhich of the following is shown'), ('C', '\n\nWhich of the following is shown')]


  [gemma-3-4b-pt] short_question   acc=0.000  unparsed=24/24  examples=[('A', '\n\nWhat is in the image?'), ('A', '\n\nWhat is the model of the'), ('C', '\n\nWhat is in the image?')]


  [gemma-3-4b-pt] newline_list     acc=0.000  unparsed=24/24  examples=[('A', '\n\nWhat is in the image?'), ('A', '\n\nWhat is in the image?'), ('C', '\n\nWhat is in the image?')]


  [gemma-3-4b-pt] bare_options     acc=0.208  unparsed=10/24  examples=[('A', '\n\n-\n\n\n'), ('A', '\n\nA=lorikeet /'), ('C', '\n\nA=home theater / B')]


  [gemma-3-4b-pt] minimal          acc=0.083  unparsed=17/24  examples=[('A', '\n\n\n\n'), ('A', '\n\n\n\n'), ('C', '\n\n\n\n')]


## Summary table

In [9]:
rows = []
for model_kind, variant_results in all_results.items():
    for variant_name, r in variant_results.items():
        rows.append({"model": model_kind, "prompt_variant": variant_name,
                     "accuracy": r["accuracy"], "unparsed": r["unparsed"]})
df = pd.DataFrame(rows)
pivot = df.pivot(index="prompt_variant", columns="model", values="accuracy")
pivot = pivot.reindex(list(PROMPT_VARIANTS.keys()))
pd.set_option("display.width", 140)
print("Accuracy by prompt variant x model (chance = 0.25):\n")
print(pivot.round(3).to_string())

print("\nVariants where ALL models beat chance (>0.25):")
above_chance_all = pivot[(pivot > 0.25).all(axis=1)]
print(above_chance_all.round(3).to_string() if len(above_chance_all) else "  (none)")

print("\nMean accuracy per variant across models:")
print(pivot.mean(axis=1).round(3).sort_values(ascending=False).to_string())

Accuracy by prompt variant x model (chance = 0.25):

model             gemma-3-4b-it  gemma-3-4b-pt  llava-1.5-7b  llava-1.6-vicuna-7b
prompt_variant                                                                   
original_verbose          0.250          0.000         0.458                0.417
short_question            0.333          0.000         0.375                0.208
newline_list              0.042          0.000         0.292                0.333
bare_options              0.042          0.208         0.292                0.250
minimal                   0.042          0.083         0.292                0.292

Variants where ALL models beat chance (>0.25):
  (none)

Mean accuracy per variant across models:
prompt_variant
original_verbose    0.281
short_question      0.229
bare_options        0.198
minimal             0.177
newline_list        0.167


## Result

**No prompt variant gets all four models above chance (0.25) simultaneously.** The
hypothesis that a simpler/shorter prompt would help was not confirmed — in fact the
opposite happened for several models:

| variant | gemma-3-4b-it | gemma-3-4b-pt | llava-1.5-7b | llava-1.6-vicuna-7b | mean |
|---|---|---|---|---|---|
| original_verbose (baseline) | 0.250 | 0.000 | **0.458** | **0.417** | **0.281** |
| short_question | **0.333** | 0.000 | 0.375 | 0.208 | 0.229 |
| newline_list | 0.042 | 0.000 | 0.292 | 0.333 | 0.167 |
| bare_options | 0.042 | 0.208 | 0.292 | 0.250 | 0.198 |
| minimal | 0.042 | 0.083 | 0.292 | 0.292 | 0.177 |

Key findings:

1. **The original verbose format is actually the best on average (0.281)**, not the
   simplified ones. For both LLaVA checkpoints, it is the single best-performing variant
   (0.458 and 0.417) — shortening the prompt *hurt* LLaVA's accuracy, it did not help.
2. **Simplification broke Gemma-3-4B-it specifically**: with terser formats
   (`newline_list`, `bare_options`, `minimal`) it stopped answering with a single letter
   at all and instead wrote full explanations ("Here's the breakdown of the images...",
   "Let's analyze each image...") — 18-23 of 24 samples became unparsed. The verbose,
   fully-worded original format is what keeps Gemma's instruct-tuned chat behavior
   constrained to a short answer.
3. **Gemma-3-4B-pt (base) cannot do this task in any format.** It never learned to
   follow an instruction to answer with a letter (unsurprising — it has no
   instruction-tuning at all), and instead continues/repeats the prompt text
   regardless of phrasing. This is a capability gap, not a prompt-format problem, and no
   prompt rewrite can fix it.

**Conclusion**: the original verbose MCQ prompt (already used in `mcq_causal_intervention.ipynb`
and `prompt_phrasing_vis_head_vs_causal.ipynb` for the Qwen-VL family) was not actually
the bottleneck for LLaVA/Gemma — it is in fact the best format tested for LLaVA, and
removing wording only broke Gemma's instruction-following further. The at-chance/below-chance
MCQ results found earlier for these families reflect a genuine visual-grounding or
instruction-following capability limitation, not an artifact of prompt complexity. This
confirms the earlier decision to use free-form generation + semantic similarity
(`steer_llava_family.ipynb`) as the causal-evaluation protocol for these families, rather
than continuing to search for a better MCQ prompt.